Load the Gold table 

In [0]:
df_gold = spark.table("retail_project.gold.sales_features")

# Start with a single store for the initial model build
store_id = 1
df_store1 = df_gold.filter(f"Store = {store_id}").orderBy("Date")

print("Rows for Store 1:", df_store1.count())
display(df_store1)

Convert to pandas

In [0]:
pdf_store1 = df_store1.select("Date", "Sales", "Open", "Promo", "TemperatureMean", "PrecipitationSum").toPandas()
pdf_store1 = pdf_store1.sort_values("Date").reset_index(drop=True)

print(pdf_store1.shape)
pdf_store1.head()

Install prophet

In [0]:
%pip install prophet

Prepare data in Prophet's required format

In [0]:
prophet_df_open = pdf_store1[pdf_store1["Open"] == 1][["Date", "Sales"]].rename(
    columns={"Date": "ds", "Sales": "y"}
)

print(prophet_df_open.shape)
prophet_df_open.head()

Train/test split(time-based, never random for time series)

In [0]:
# Calculate how many calendar days needed to cover the full test date range
calendar_days_needed = (test_df_open["ds"].max() - train_df_open["ds"].max()).days

future = model.make_future_dataframe(periods=calendar_days_needed)
forecast = model.predict(future)

forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(10)

Evaluation

In [0]:
eval_df = test_df_open.merge(forecast[["ds", "yhat"]], on="ds", how="left")

print(eval_df.isna().sum())
eval_df.head(10)

In [0]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import numpy as np

mape = mean_absolute_percentage_error(eval_df["y"], eval_df["yhat"])
rmse = np.sqrt(mean_squared_error(eval_df["y"], eval_df["yhat"]))

print(f"MAPE: {mape:.2%}")
print(f"RMSE: {rmse:.2f}")

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(eval_df["ds"], eval_df["y"], label="Actual", marker="o")
plt.plot(eval_df["ds"], eval_df["yhat"], label="Predicted", marker="x")
plt.title(f"Store {store_id} - Actual vs Predicted Sales (Prophet)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Add regressors

In [0]:
# Rebuild prophet_df_open WITH extra regressor columns
prophet_df_open = pdf_store1[pdf_store1["Open"] == 1][
    ["Date", "Sales", "Promo", "TemperatureMean", "PrecipitationSum"]
].rename(columns={"Date": "ds", "Sales": "y"})

split_date = prophet_df_open["ds"].max() - pd.Timedelta(days=42)
train_df_open = prophet_df_open[prophet_df_open["ds"] <= split_date]
test_df_open = prophet_df_open[prophet_df_open["ds"] > split_date]

model_v2 = Prophet()
model_v2.add_regressor("Promo")
model_v2.add_regressor("TemperatureMean")
model_v2.add_regressor("PrecipitationSum")

model_v2.fit(train_df_open)

In [0]:
future_v2 = test_df_open[["ds", "Promo", "TemperatureMean", "PrecipitationSum"]]
forecast_v2 = model_v2.predict(future_v2)

eval_df_v2 = test_df_open.merge(forecast_v2[["ds", "yhat"]], on="ds", how="left")

mape_v2 = mean_absolute_percentage_error(eval_df_v2["y"], eval_df_v2["yhat"])
rmse_v2 = np.sqrt(mean_squared_error(eval_df_v2["y"], eval_df_v2["yhat"]))

print(f"MAPE (with regressors): {mape_v2:.2%}")
print(f"RMSE (with regressors): {rmse_v2:.2f}")

Plot both models visually

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(eval_df["ds"], eval_df["y"], label="Actual", marker="o", color="black")
plt.plot(eval_df["ds"], eval_df["yhat"], label="Prophet (baseline)", marker="x", linestyle="--")
plt.plot(eval_df_v2["ds"], eval_df_v2["yhat"], label="Prophet + Promo/Weather", marker="s", linestyle="--")
plt.title(f"Store {store_id} - Actual vs Predicted Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()